# ChatPromptTemplate 고급 활용 — MessagesPlaceholder와 멀티턴 대화

Ch01에서 LLM 호출을 배웠다면, 이제 프롬프트를 체계적으로 관리하는 법을 배워요.
`01-PromptTemplate`에서 기본 `ChatPromptTemplate` 사용법을 익혔다면, 이 노트북에서는 **대화 이력**을 동적으로 다루는 방법을 배워요.

실제 챗봇은 이전 대화를 기억해야 해요. `MessagesPlaceholder`(메시지 플레이스홀더)가 그 역할을 담당합니다.

## 학습 목표

- `MessagesPlaceholder`로 가변 길이 대화 이력을 프롬프트에 삽입할 수 있어요
- `HumanMessage` / `AIMessage`로 대화 이력 리스트를 직접 구성할 수 있어요
- 멀티턴(multi-turn) 대화 시스템을 설계하고 이력 길이를 제한하는 방법을 알 수 있어요
- 공통 시스템 메시지를 재사용해 여러 프롬프트를 효율적으로 관리할 수 있어요

## 사전 지식

- `01-PromptTemplate` 노트북의 `ChatPromptTemplate` 기본 사용법
- LCEL 파이프라인 (`|` 연산자) 기초

---

> **ChatPromptTemplate 메시지 구성도** — 아래 다이어그램은 System, Human, AI 메시지와 대화 이력이 어떻게 조합되어 LLM에 전달되는지 보여줘요.

```mermaid
flowchart TD
    SYS["SystemMessage<br/>'당신은 AI 어시스턴트입니다.'"]:::process
    HIST["MessagesPlaceholder<br/>chat_history<br/>HumanMessage, AIMessage, ...<br/>이전 대화 이력 (가변 길이)"]:::storage
    HUM["HumanMessage<br/>현재 질문 {question}"]:::input
    LIMIT["이력 길이 제한<br/>최근 N개 메시지만 전달<br/>토큰 비용 제어"]:::error
    FINAL["ChatPromptTemplate<br/>메시지 배열 조립"]:::process
    LLM["LLM"]:::external
    RES["AIMessage<br/>응답 (맥락 반영)"]:::output

    SYS --> FINAL
    HIST -->|"전체 또는 최근 N개"| LIMIT
    LIMIT --> FINAL
    HUM --> FINAL
    FINAL --> LLM
    LLM --> RES

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

In [1]:
# 필수 라이브러리 import
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 모델 초기화
# temperature=0: 일관된 출력을 위해 설정
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. MessagesPlaceholder — 동적 대화 이력 삽입

`MessagesPlaceholder`(메시지 플레이스홀더)는 가변 길이의 메시지 리스트를 프롬프트에 삽입할 수 있게 해줘요.

> **실무 팁:** 대화 이력이 길어질수록 토큰 사용량이 늘어나요. 프로덕션 챗봇에서는 `chat_history[-10:]`처럼 최근 N개만 전달하거나, 오래된 대화를 요약해서 압축하는 전략을 사용해요.

### 언제 사용할까요?

- 멀티턴 대화 시스템 (챗봇, 대화형 에이전트)
- 이전 대화 내용을 기억해야 하는 경우
- 컨텍스트를 유지하는 QA 시스템

### 주요 메시지 타입

| 클래스 | 역할 | 사용 시점 |
|--------|------|----------|
| `SystemMessage` | LLM 행동 방식·규칙 정의 | 프롬프트 맨 앞 |
| `HumanMessage` | 사용자 발화 | 대화 이력에 추가 |
| `AIMessage` | LLM 이전 응답 | 대화 이력에 추가 |

In [ ]:
# 시나리오: 이전 대화 내용을 기억하는 챗봇

# 1단계: MessagesPlaceholder를 포함한 ChatPromptTemplate 정의
# MessagesPlaceholder: 가변 길이의 메시지 리스트를 프롬프트 중간에 삽입하는 플레이스홀더
#   - variable_name: invoke() 호출 시 전달할 딕셔너리 키 이름


# 2단계: 체인 구성


# 3단계: 첫 번째 대화 (이력 없음)


# 4단계: 두 번째 대화 (이력 포함)
# HumanMessage / AIMessage: 이전 대화를 메시지 객체로 표현



## 2. 복잡한 대화 시스템 — 고객 지원 챗봇

실제 업무에서 사용할 수 있는 멀티턴 대화 시스템을 구축해볼게요.

> **주의:** 챗봇에서 `chat_history`를 관리할 때는 매 턴마다 `HumanMessage`와 `AIMessage`를 쌍으로 추가해야 해요. 하나만 추가하면 역할 순서가 어긋나 LLM이 혼란스러워할 수 있어요.

In [ ]:
# 시나리오: 제품 문의를 처리하는 고객 지원 챗봇

# 1단계: 고객 지원 챗봇 프롬프트 정의


# 2단계: 체인 구성


# 3단계: 대화 시뮬레이션
# chat_history: 매 턴마다 HumanMessage + AIMessage 쌍을 추가해 대화 맥락 유지



## 3. 템플릿 재사용 — 공통 시스템 메시지 공유하기

번역·요약·교정처럼 역할만 다르고 지침 구조가 같은 작업이라면, `SystemMessagePromptTemplate`을 하나 정의해두고 여러 프롬프트에서 공유할 수 있어요.

> **실무 팁:** 공통 시스템 메시지를 변수로 분리해두면, 회사 정책이 바뀌었을 때 한 곳만 수정하면 모든 체인에 반영돼요. 유지보수 비용이 크게 줄어들어요.

In [ ]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 시나리오: 여러 프롬프트에서 동일한 시스템 메시지 재사용

# 1단계: 공통 시스템 메시지 템플릿 정의
# SystemMessagePromptTemplate: 시스템 역할을 변수로 분리하여 재사용 가능하게 구성


# 2단계: 번역 프롬프트에 재사용
# HumanMessagePromptTemplate: 사용자 질문 부분을 독립적으로 정의


# 3단계: 요약 프롬프트에 재사용
# 동일한 common_system을 재사용 → 회사 정책 변경 시 한 곳만 수정하면 됨


# 4단계: 번역 실행


# 5단계: 요약 실행



## 💡 핵심 정리

### MessagesPlaceholder 사용법

```python
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# MessagesPlaceholder 포함 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# 대화 이력 전달
chain.invoke({
    "chat_history": [
        HumanMessage(content="안녕하세요"),
        AIMessage(content="안녕하세요! 무엇을 도와드릴까요?")
    ],
    "question": "날씨 알려주세요"
})
```

### 주요 메시지 타입

| 메시지 타입 | 설명 | 사용 시점 |
|-----------|------|-----------|
| `SystemMessage` | 시스템 지침 | LLM 역할/규칙 정의 |
| `HumanMessage` | 사용자 메시지 | 사용자 질문/요청 |
| `AIMessage` | AI 응답 | LLM의 이전 응답 |

### 활용 시나리오

**언제 사용하면 좋을까요?**
- ✅ 멀티턴 대화 시스템 (챗봇, 대화형 에이전트)
- ✅ 이전 대화를 기억해야 하는 경우
- ✅ 컨텍스트가 중요한 QA 시스템
- ✅ 고객 지원, 상담 챗봇

**주의사항**:
- ⚠️ 대화 이력이 길어지면 토큰 사용량 증가 → 비용 상승
- ⚠️ 적절한 이력 길이 제한 필요 (최근 N개만 유지)
- ⚠️ 메시지 타입을 정확히 지정 (HumanMessage, AIMessage)

### 대화 이력 관리 팁

```python
# 최근 N개 메시지만 유지
MAX_HISTORY = 10
chat_history = chat_history[-MAX_HISTORY:]

# 토큰 수로 제한 (tiktoken 사용)
# 토큰 수가 제한을 초과하면 오래된 메시지부터 제거
```


## 연습 문제

직접 해보면서 학습 내용을 정리해봅시다!

### 연습 1: 멀티턴 기술 면접 시뮬레이터

**과제**: `MessagesPlaceholder`를 사용하여 기술 면접을 시뮬레이션하는 대화 시스템을 만드세요.

**요구사항**:
- `ChatPromptTemplate` + `MessagesPlaceholder` 사용
- 시스템 메시지: 면접관 역할 정의 (Python 시니어 개발자 면접관)
- 3턴 이상의 대화를 시뮬레이션 (질문 -> 답변 -> 후속 질문)
- 대화 이력(`chat_history`)을 매 턴마다 업데이트

**힌트**: 면접관이 이전 답변을 바탕으로 후속 질문을 하도록 대화 이력을 활용하세요.

In [5]:
# ============================================================
# TODO: MessagesPlaceholder를 사용하여 멀티턴 기술 면접 시뮬레이터를 만드세요
# 힌트: ChatPromptTemplate.from_messages([("system", ...), MessagesPlaceholder(...), ("human", ...)])로 구성하고
#       매 턴마다 chat_history에 HumanMessage + AIMessage 쌍을 추가하여 이전 답변을 유지하세요
# 예상 결과: 면접관이 이전 답변을 바탕으로 후속 질문을 하는 3턴 이상의 대화
# ============================================================

# 여기에 코드를 작성하세요

# interview_prompt = ChatPromptTemplate.from_messages([
#     ("system", "..."),
#     MessagesPlaceholder(variable_name="chat_history"),
#     ("human", "{answer}")
# ])

### 연습 1 풀이

In [ ]:
# 연습 1 풀이: 멀티턴 기술 면접 시뮬레이터



### 연습 2: 재사용 가능한 다목적 분석 시스템

**과제**: 공통 시스템 메시지 템플릿을 재사용하여 여러 분석 작업(감정 분석, 키워드 추출)을 수행하는 시스템을 만드세요.

**요구사항**:
- `SystemMessagePromptTemplate`과 `HumanMessagePromptTemplate`을 사용
- 공통 시스템 메시지를 정의하고 두 가지 이상의 분석 프롬프트에서 재사용
- 변수: `role`, `output_format`, `text`
- 감정 분석 체인과 키워드 추출 체인을 각각 구성하여 실행

**힌트**: 공통 시스템 메시지에서 `role`과 `output_format`을 변수로 사용하면 다양한 분석에 재사용할 수 있습니다.

In [7]:
# ============================================================
# TODO: 공통 SystemMessagePromptTemplate을 재사용하여 감정 분석과 키워드 추출 체인을 만드세요
# 힌트: SystemMessagePromptTemplate.from_template(...)으로 role과 output_format을 변수로 갖는
#       공통 시스템 메시지를 정의하고, 두 프롬프트에서 동일한 객체를 재사용하세요
# 예상 결과: 같은 텍스트에 대해 감정 분석 결과와 키워드 추출 결과를 각각 출력
# ============================================================

# 여기에 코드를 작성하세요

# common_system = SystemMessagePromptTemplate.from_template(
#     "당신은 {role}입니다. 결과를 {output_format} 형식으로 출력합니다."
# )
#
# analysis_human = HumanMessagePromptTemplate.from_template(
#     "다음 텍스트를 분석해주세요:\n{text}"
# )

### 연습 2 풀이

In [ ]:
# 연습 2 풀이: 재사용 가능한 다목적 분석 시스템



### 연습 3: 대화 이력 길이 제한이 있는 챗봇

**과제**: 대화 이력을 최근 N개 메시지로 제한하는 챗봇을 만드세요.

**요구사항**:
- `MessagesPlaceholder`를 사용한 대화형 시스템
- 대화 이력을 최근 4개 메시지(2턴)로 제한
- 5턴 이상의 대화를 시뮬레이션하여 이력 제한이 동작하는지 확인
- 시스템 메시지: 여행 가이드 AI 역할

**힌트**: `chat_history[-MAX_MESSAGES:]` 슬라이싱을 사용하면 최근 메시지만 유지할 수 있습니다.

In [9]:
# ============================================================
# TODO: 대화 이력을 최근 4개 메시지로 제한하는 여행 가이드 챗봇을 만드세요
# 힌트: chat_history[-MAX_MESSAGES:]로 슬라이싱하여 최근 메시지만 LLM에 전달하고
#       전체 이력(chat_history)에는 모든 메시지를 계속 누적하세요
# 예상 결과: 5턴 이후 초기 대화 내용을 기억하지 못하는 이력 제한 동작 확인
# ============================================================

# 여기에 코드를 작성하세요

# MAX_MESSAGES = 4  # 최근 4개 메시지(2턴)만 유지
#
# travel_prompt = ChatPromptTemplate.from_messages([
#     ("system", "..."),
#     MessagesPlaceholder(variable_name="chat_history"),
#     ("human", "{question}")
# ])

### 연습 3 풀이

In [ ]:
# 연습 3 풀이: 대화 이력 길이 제한이 있는 챗봇



## 마무리 — 이번 노트북에서 배운 것

| 개념 | 핵심 클래스 | 언제 쓰나요? |
|------|------------|-------------|
| 동적 대화 이력 삽입 | `MessagesPlaceholder` | 멀티턴 챗봇, 이전 대화를 기억해야 할 때 |
| 대화 이력 구성 | `HumanMessage`, `AIMessage` | 이전 발화를 메시지 객체로 표현할 때 |
| 이력 길이 제한 | `history[-N:]` 슬라이싱 | 토큰 비용을 제어하고 싶을 때 |
| 템플릿 재사용 | `SystemMessagePromptTemplate` | 공통 지침을 여러 프롬프트에서 공유할 때 |

`MessagesPlaceholder`는 단순한 프롬프트를 진짜 대화 시스템으로 끌어올리는 핵심 도구예요. 대화 이력을 어떻게 관리하느냐에 따라 챗봇의 품질이 크게 달라집니다.

---

**다음 챕터 예고:** `03_OutputParser` — LLM이 반환하는 자유 형식 텍스트를 JSON, 리스트 등 구조화된 데이터로 파싱하는 방법을 배워요. 프롬프트 → LLM → 파싱까지 이어지는 파이프라인이 완성됩니다.